In [0]:
# Passo 2.1 - Teste inicial da API do SICONFI (DCA - Despesas por Função)
import requests

url = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca"
params = {
    "an_exercicio": 2025,
    "no_anexo": "DCA-Anexo I-E",   # I-E = Despesas Orçamentárias por Função
    "id_ente": 3550308             # código IBGE de São Paulo (só para teste)
}

resposta = requests.get(url, params=params)
dados = resposta.json()
print(f"Total de linhas: {len(dados['items'])}")
dados["items"][:3]

In [0]:
# Passo 2.2 - Verificando os estágios de despesa disponíveis e os rótulos de conta
import pandas as pd

df = pd.DataFrame(dados["items"])
print(df["coluna"].unique())        # deve mostrar Empenhada / Liquidada / Paga
print(df["conta"].unique())    # amostra dos rótulos, pra visualizar o padrão função x subfunção

In [0]:
# Passo 2.3 - Isolando despesas por função, em Despesas Liquidadas
import re

liquidadas = df[df["coluna"] == "Despesas Liquidadas"].copy()

# Função = código de 2 dígitos seguido de " - " (sem ponto antes, o que exclui subfunções)
padrao_funcao = re.compile(r'^\d{2} - ')
funcoes = liquidadas[liquidadas["conta"].str.match(padrao_funcao)]

funcoes = funcoes[["conta", "valor", "populacao"]].sort_values("valor", ascending=False)
funcoes

In [0]:
# Passo 2.4 - Buscando a Receita Total (Anexo I-C), mesmo município/ano
params_receita = {
    "an_exercicio": 2025,
    "no_anexo": "DCA-Anexo I-C",
    "id_ente": 3550308
}

resposta_receita = requests.get(url, params=params_receita)
dados_receita = resposta_receita.json()

df_receita = pd.DataFrame(dados_receita["items"])
print(df_receita["coluna"].unique())
print(df_receita["conta"].unique()[:15])

In [0]:
# Passo 2.5 - Isolando a Receita Total
receitas_realizadas = df_receita[df_receita["coluna"] == "Receitas Brutas Realizadas"]

receita_total = receitas_realizadas[
    receitas_realizadas["conta"] == "RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I)"
]

receita_total[["conta", "valor"]]

In [0]:
# Passo 2.6 - Calculando o % da receita gasto em cada função
receita_valor = receita_total["valor"].iloc[0]

funcoes["pct_receita"] = (funcoes["valor"] / receita_valor * 100).round(2)
funcoes[["conta", "valor", "pct_receita"]].sort_values("pct_receita", ascending=False)

In [0]:
# Passo 2.7 - Somando o percentual da receita gasto em todas as funções
soma_pct = funcoes["pct_receita"].sum()
print(f"Soma do % da receita: {soma_pct:.2f}%")

In [0]:
# Passo 2.7 - Buscando a lista completa de municípios do IBGE
url_municipios = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
resposta_mun = requests.get(url_municipios)
municipios = resposta_mun.json()

print(f"Total de municípios: {len(municipios)}")

import json
print(json.dumps(municipios[0], indent=2, ensure_ascii=False))  # olha a estrutura completa do primeiro item

In [0]:
# Passo 2.8 - Achatando a lista de municípios (id, nome, UF, região)
lista_municipios = []
for m in municipios:
    # Alguns municípios têm microrregiao=None, usar regiao-imediata como alternativa
    if m["microrregiao"] is not None:
        uf = m["microrregiao"]["mesorregiao"]["UF"]
    else:
        uf = m["regiao-imediata"]["regiao-intermediaria"]["UF"]
    
    lista_municipios.append({
        "cod_ibge": m["id"],
        "municipio": m["nome"],
        "uf": uf["sigla"],
        "regiao": uf["regiao"]["nome"]
    })

df_municipios = pd.DataFrame(lista_municipios)
print(df_municipios.shape)
df_municipios["regiao"].value_counts()

In [0]:
# Passo 2.9 - Testando se a API retorna todos os municípios de uma vez (sem id_ente)
params_teste = {
    "an_exercicio": 2025,
    "no_anexo": "DCA-Anexo I-E"
}
resposta_teste = requests.get(url, params=params_teste)

print("status:", resposta_teste.status_code)
dados_teste = resposta_teste.json()
print("itens nesta página:", len(dados_teste.get("items", [])))
print("chaves da resposta:", dados_teste.keys())
print("hasMore:", dados_teste.get("hasMore"))
print("limit:", dados_teste.get("limit"), "| offset:", dados_teste.get("offset"))

In [0]:
# Passo 2.10 - Testando o tempo de resposta com uma amostra pequena
import time

amostra = df_municipios["cod_ibge"].head(5).tolist()

inicio = time.time()
for codigo in amostra:
    params = {"an_exercicio": 2025, "no_anexo": "DCA-Anexo I-E", "id_ente": codigo}
    r = requests.get(url, params=params)
fim = time.time()

tempo_por_chamada = (fim - inicio) / len(amostra)
print(f"Tempo médio por chamada: {tempo_por_chamada:.2f}s")
print(f"Estimativa para 5.571 municípios x 2 chamadas (despesa+receita): {tempo_por_chamada * 5571 * 2 / 60:.1f} minutos")

In [0]:
%pip install xlrd==1.2.0

In [0]:
# Passo 2.11 - Buscando as estimativas de população do IBGE (2025)
# Como o arquivo é .xls (formato antigo), usar xlrd diretamente ao invés de pd.read_excel
import xlrd
import io

url_populacao = "https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2025/POP2025_20260828.xls"

# Baixar o arquivo
response = requests.get(url_populacao)
file_content = io.BytesIO(response.content)

# Ler com xlrd
workbook = xlrd.open_workbook(file_contents=file_content.read())

# Converter cada aba para DataFrame
df_pop_raw = {}
for sheet_name in workbook.sheet_names():
    sheet = workbook.sheet_by_name(sheet_name)
    data = [sheet.row_values(i) for i in range(sheet.nrows)]
    df_pop_raw[sheet_name] = pd.DataFrame(data[1:], columns=data[0])  # primeira linha como cabeçalho

print(f"Abas encontradas: {list(df_pop_raw.keys())}")

In [0]:
# Verificar a estrutura de cada aba
for nome_aba, df in df_pop_raw.items():
    print(f"\n{'='*60}")
    print(f"Aba: {nome_aba}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)[:10]}...")  # primeiras 10 colunas
    print(f"\nPrimeiras 3 linhas:")
    print(df.head(3).to_string())

In [0]:
# Passo 2.12 - Explorando a aba "Municípios"
df_pop_mun = df_pop_raw["Municípios"]
print(df_pop_mun.shape)
df_pop_mun.head(10)

In [0]:
# Passo 2.13 - Corrigindo o cabeçalho (pulando a linha de título)
sheet = workbook.sheet_by_name("Municípios")
data = [sheet.row_values(i) for i in range(sheet.nrows)]

# Linha 0 = título solto, linha 1 = cabeçalho real, dados a partir da linha 2
df_pop_mun = pd.DataFrame(data[2:], columns=data[1])
print(df_pop_mun.shape)
df_pop_mun.head()
print(df_pop_mun.tail())  # importante: ver se tem lixo de rodapé (notas explicativas) nas últimas linhas
df_pop_mun.columns

In [0]:
# Passo 2.14 - Checando linhas que não são municípios de verdade
print(df_pop_mun["UF"].unique())  # deve ter só as 27 siglas de estado, se tiver algo mais é sinal de sujeira

linhas_suspeitas = df_pop_mun[df_pop_mun["NOME DO MUNICÍPIO"].isna() | (df_pop_mun["NOME DO MUNICÍPIO"] == "")]
print(linhas_suspeitas.shape)
linhas_suspeitas

In [0]:
# Passo 2.15 - Removendo o rodapé e criando o código IBGE de 7 dígitos
df_pop_mun_limpo = df_pop_mun[
    df_pop_mun["NOME DO MUNICÍPIO"].notna() & (df_pop_mun["NOME DO MUNICÍPIO"] != "")
].copy()

# Código IBGE = 2 dígitos da UF + 5 dígitos do município
# xlrd lê números como float (ex: 50308.0), então convertemos para int antes de str
# para evitar sufixo ".0" que quebraria o cast final
df_pop_mun_limpo["cod_ibge"] = (
    df_pop_mun_limpo["COD. UF"].astype(int).astype(str).str.zfill(2) +
    df_pop_mun_limpo["COD. MUNIC"].astype(int).astype(str).str.zfill(5)
).astype(int)

df_pop_mun_limpo["populacao_2025"] = df_pop_mun_limpo["POPULAÇÃO ESTIMADA"].astype(int)
df_pop_mun_limpo = df_pop_mun_limpo[["cod_ibge", "NOME DO MUNICÍPIO", "UF", "populacao_2025"]]

print(df_pop_mun_limpo.shape)
df_pop_mun_limpo.head()

# Conferindo se São Paulo está lá certinho
df_pop_mun_limpo[df_pop_mun_limpo["cod_ibge"] == 3550308]

In [0]:
# Passo 2.16 - Classificando os municípios por porte populacional
bins = [0, 20000, 100000, 500000, float("inf")]
labels = ["Pequeno (até 20 mil)", "Médio (20 mil a 100 mil)", "Grande (100 mil a 500 mil)", "Metrópole (acima de 500 mil)"]

df_pop_mun_limpo["porte"] = pd.cut(df_pop_mun_limpo["populacao_2025"], bins=bins, labels=labels)

df_pop_mun_limpo["porte"].value_counts()

In [0]:
# Passo 2.17 - Identificando as capitais
capitais_por_uf = {
    "AC": "Rio Branco", "AL": "Maceió", "AP": "Macapá", "AM": "Manaus",
    "BA": "Salvador", "CE": "Fortaleza", "DF": "Brasília", "ES": "Vitória",
    "GO": "Goiânia", "MA": "São Luís", "MT": "Cuiabá", "MS": "Campo Grande",
    "MG": "Belo Horizonte", "PA": "Belém", "PB": "João Pessoa", "PR": "Curitiba",
    "PE": "Recife", "PI": "Teresina", "RJ": "Rio de Janeiro", "RN": "Natal",
    "RS": "Porto Alegre", "RO": "Porto Velho", "RR": "Boa Vista", "SC": "Florianópolis",
    "SP": "São Paulo", "SE": "Aracaju", "TO": "Palmas"
}

df_pop_mun_limpo["eh_capital"] = df_pop_mun_limpo.apply(
    lambda row: capitais_por_uf.get(row["UF"]) == row["NOME DO MUNICÍPIO"], axis=1
)

print("Capitais encontradas:", df_pop_mun_limpo["eh_capital"].sum())  # deveria dar 27
df_pop_mun_limpo[df_pop_mun_limpo["eh_capital"]][["cod_ibge", "NOME DO MUNICÍPIO", "UF", "populacao_2025"]]

In [0]:
# Passo 2.18 (corrigido) - Convertendo para texto antes de marcar as capitais
df_pop_mun_limpo["porte"] = df_pop_mun_limpo["porte"].astype(str)
df_pop_mun_limpo.loc[df_pop_mun_limpo["eh_capital"], "porte"] = "Capital"

df_pop_mun_limpo["porte"].value_counts()

In [0]:
# Passo 2.19 - Juntando a região à tabela de população/porte
df_amostragem = df_pop_mun_limpo.merge(
    df_municipios[["cod_ibge", "regiao"]],
    on="cod_ibge",
    how="left"
)

print("Municípios sem região após o merge:", df_amostragem["regiao"].isna().sum())  # deveria ser 0
df_amostragem.head()

In [0]:
# Passo 2.20 - Montando a amostra estratificada por porte x região
import numpy as np
np.random.seed(42)

# Capitais e Metrópoles entram por completo
amostra_completa = df_amostragem[df_amostragem["porte"].isin(["Capital", "Metrópole (acima de 500 mil)"])]

# Das demais faixas, sorteamos até 15 por combinação de porte x região
N_POR_GRUPO = 30
amostra_estratificada = (
    df_amostragem[~df_amostragem["porte"].isin(["Capital", "Metrópole (acima de 500 mil)"])]
    .groupby(["porte", "regiao"], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), N_POR_GRUPO), random_state=42))
)

amostra_final = pd.concat([amostra_completa, amostra_estratificada]).reset_index(drop=True)

print(f"Total de municípios na amostra: {len(amostra_final)}")
print(amostra_final["porte"].value_counts())
print(f"Tempo estimado de coleta: {len(amostra_final) * 0.87 * 2 / 60:.1f} minutos")

In [0]:
# Passo 2.21 - Transformando a exploração em função reutilizável (despesas por função)
def buscar_despesas_funcao(cod_ibge, ano=2025):
    params = {"an_exercicio": ano, "no_anexo": "DCA-Anexo I-E", "id_ente": cod_ibge}
    resp = requests.get(url, params=params)
    dados = resp.json()

    df_temp = pd.DataFrame(dados.get("items", []))
    if df_temp.empty:
        return pd.DataFrame()  # município sem dado disponível para esse ano

    liquidadas = df_temp[df_temp["coluna"] == "Despesas Liquidadas"]
    funcoes = liquidadas[liquidadas["conta"].str.match(r'^\d{2} - ')].copy()
    funcoes["cod_ibge"] = cod_ibge
    return funcoes[["cod_ibge", "conta", "valor"]]

# Testando com São Paulo, para garantir que a função reproduz o resultado que já validamos
teste = buscar_despesas_funcao(3550308)
print(teste.shape)
teste.head()

In [0]:
# Passo 2.22 - Função reutilizável para a receita total
def buscar_receita_total(cod_ibge, ano=2025):
    params = {"an_exercicio": ano, "no_anexo": "DCA-Anexo I-C", "id_ente": cod_ibge}
    resp = requests.get(url, params=params)
    dados = resp.json()

    df_temp = pd.DataFrame(dados.get("items", []))
    if df_temp.empty:
        return None

    total = df_temp[
        (df_temp["coluna"] == "Receitas Brutas Realizadas") &
        (df_temp["conta"] == "RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I)")
    ]
    if total.empty:
        return None
    return total["valor"].iloc[0]

# Testando com São Paulo
teste_receita = buscar_receita_total(3550308)
print(teste_receita)

In [0]:
# Passo 2.23 - Coleta em massa para os 486 municípios da amostra
import time

lista_despesas = []
lista_receitas = []
erros = []

codigos = amostra_final["cod_ibge"].tolist()
inicio = time.time()

for i, codigo in enumerate(codigos):
    try:
        despesas = buscar_despesas_funcao(codigo)
        if not despesas.empty:
            lista_despesas.append(despesas)

        receita = buscar_receita_total(codigo)
        lista_receitas.append({"cod_ibge": codigo, "receita_total": receita})

    except Exception as e:
        erros.append({"cod_ibge": codigo, "erro": str(e)})

    if (i + 1) % 50 == 0:
        decorrido = time.time() - inicio
        print(f"{i+1}/{len(codigos)} municípios processados ({decorrido:.0f}s)")

print(f"\nConcluído em {(time.time() - inicio) / 60:.1f} minutos")
print(f"Municípios com erro (falha na chamada): {len(erros)}")

df_despesas_bruto = pd.concat(lista_despesas, ignore_index=True)
df_receitas_bruto = pd.DataFrame(lista_receitas)

print("Despesas:", df_despesas_bruto.shape)
print("Receitas:", df_receitas_bruto.shape)
print("Municípios sem receita retornada:", df_receitas_bruto["receita_total"].isna().sum())

In [0]:
# Passo 2.24 - Investigando os municípios sem receita retornada
sem_receita = df_receitas_bruto[df_receitas_bruto["receita_total"].isna()]
sem_receita = sem_receita.merge(
    df_amostragem[["cod_ibge", "NOME DO MUNICÍPIO", "UF", "porte"]],
    on="cod_ibge"
)
sem_receita

In [0]:
# Passo 2.25 - Verificando os catálogos disponíveis
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
# Passo 2.26 - Criando o schema do projeto e salvando as tabelas Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gastos_municipais")

# Limpando nomes de coluna antes de salvar (Delta não aceita espaços/acentos)
df_amostragem_limpo = df_amostragem.rename(columns={
    "NOME DO MUNICÍPIO": "municipio",
    "UF": "uf"
})[["cod_ibge", "municipio", "uf", "populacao_2025", "porte", "regiao", "eh_capital"]]

# Salvando as tabelas Bronze (dado bruto, como veio da fonte)
spark.createDataFrame(df_despesas_bruto).write.mode("overwrite").saveAsTable("workspace.gastos_municipais.bronze_despesas")
spark.createDataFrame(df_receitas_bruto).write.mode("overwrite").saveAsTable("workspace.gastos_municipais.bronze_receitas")
spark.createDataFrame(df_amostragem_limpo).write.mode("overwrite").saveAsTable("workspace.gastos_municipais.bronze_municipios_amostra")

print("Tabelas salvas!")
spark.sql("SHOW TABLES IN workspace.gastos_municipais").show(truncate=False)